# Task 4 - Anomaly detection using autoencoders

### <span style="color:red">Deadline: Tuesday, July 1, 2025 at 11:59 p.m.</span>

# IMPORTANT SUBMISSION INSTRUCTIONS

- When you're done, download the notebook and rename it to task4_name.ipynb
- Only submit the `ipynb` file, no other file is required.
- The deadline is strict.
- Minimal requirement for passing: solving all code cells.

Implementation
- Do not change the cells which are marked as "Do not change", similarly write your solution to the marked cells.

# Topics

In this exercise sheet, you will:
- Work with a dataset for `dynamic behavior analysis of automotive software systems`.
- Implement autoencoders (AEs) for unsupervised anomaly detection.
- Compare MLP-based and CNN-based AEs.
- Vary architecture to improve model performance.
- Apply scheduled learning rate.
- Evaluate the performance of the model.

*We are looking forward to seeing your solutions! Have fun!*

# Related publications
 - Abboush, M.; Bamal, D.; Knieke, C.; Rausch, A. Hardware-in-the-Loop-Based Real-Time Fault Injection Framework for Dynamic Behavior Analysis of Automotive Software Systems. Sensors 2022, 22, 1360. https://doi.org/10.3390/s22041360 

 - Abboush, M.; Bamal, D.; Knieke, C.; Rausch, A. Intelligent Fault Detection and Classification Based on Hybrid Deep Learning Methods for Hardware-in-the-Loop Test of Automotive Software Systems. Sensors 2022, 22, 4066. https://doi.org/10.3390/s22114066 

## Tutorials

Some python libraries are required to accomplish the tasks assigned in this homework. If you feel like you need to follow a tutorial before, feel free to do so:
*   [Scikit-learn Tutorials](https://scikit-learn.org/stable/tutorial/index.html)
*   [TensorFlow Tutorials](https://www.tensorflow.org/tutorials)
*   [Matplotlib Tutorials](https://matplotlib.org/stable/tutorials/index.html)

## Imports

In [ ]:
import numpy as np
import pandas as pd
import glob

import tensorflow as tf
from tensorflow.keras import models, layers, optimizers, losses, callbacks
from sklearn.preprocessing import MinMaxScaler

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 24
np.random.seed(SEED)

## System checks


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
cpus = tf.config.list_physical_devices('CPU')
print(gpus)
print(cpus)

Choose your device for computation. CPU or one of your CUDA devices.

In [ ]:
tf.config.set_visible_devices(gpus, 'GPU')

# Anomaly detection via autoencoders

Anomaly detection is a technique used to identify abnormal or anomalous patterns within a dataset. Autoencoders are neural networks that are trained to reconstruct their input data, typically by learning a compressed representation of the data in an intermediate layer called the **bottleneck layer**.

Anomalies can be identified using autoencoders by comparing the input data to the reconstructed output, as anomalies often result in a higher reconstruction error. In this task, we employ autoencoders for anomaly detection in automotive
software systems.
 

# Subtask 2.1

## Loading the dataset and pre-processing

The dataset consists of time-series sensor measurements of healthy and faulty operation of the system.<br>
There are three different behaviors available for healthy data: ['JC08', 'JP10', 'FP75'].<br>
And, there are two faulty data for each healthy behavior.<br><br>

The fault type in each file is as following:

| File Prefix | Fault type| Fault Meaning|
|------|------|------|
| GainRPM | Increasing RPM by X2 |  The motor rotations per minute is not behaving normally |
| OffsetACC | Increasing Acceleration by +5 | The Gas pedal is letting more gas in than normal |

In [ ]:
# unzip data to Data folder if not extracted

# import zipfile

# with zipfile.ZipFile("Data.zip", "r") as z:
#     z.extractall("./")

In [ ]:
# list of available behaviors
data_dir = "./Data/"
behaviors = [i.split("-")[-1].removesuffix(".csv") for i in glob.glob(data_dir + f'Healthy*.csv')]
behaviors

In [ ]:
# read healthy and faulty data where a list of data frames have the healthy and faulty behavior within the same list
# this helps later when scaling and visualizing the data

dfs = []

for b in behaviors:
    dfs.append([])
    
    # read training data
    dfs[-1].append(pd.read_csv(glob.glob(data_dir + f'Healthy*{b}.csv')[0]))
    
    # read testing data
    for test_df in glob.glob(data_dir + f'[!Healthy]*{b}.csv'):
        dfs[-1].append(pd.read_csv(test_df))

In [ ]:
dfs[0][0].head()

#### TODO

- Visualize `All Features` of one healthy data and its two faulty data time-series in a grid of plots.

**Note:** Do not forget to add title, axis labels and a legend!
This applies in general, please keep in mind.

In [ ]:
####################
## YOUR CODE HERE ##
####################


## Pre-processing

#### TODO 
- Scale the data using `min-max scaler` to scale each healthy behavior and its faulty data.
- Extract the labels to `y_train` for healthy data and `y_test` for faulty data.<br>
- Do not forget to drop **"time"** and **"label"** columns before scaling



In [ ]:
X_train = []
X_test = []

y_train = []
y_test = []

for df in dfs:
    ...
    ####################
    ## YOUR CODE HERE ##
    ####################

Since we are working with time-series data, it is beneficial to consider a sequence of time steps as the input of the model. In this way it is possible to exploit and learn the temporal dependencies in the data.

To this end, let us consider a sliding window with the length of $\ell$ and slide it over the whole time series with a stride of $s$ to generate our sequences. In this way, we will have:

$ X_0 = [x_0, ..., x_{\ell}]$

$ X_1 = [x_s, ..., x_{\ell+s}]$

$ X_2 = [x_{2s}, ..., x_{\ell+2s}]$

...

$ X_n = [x_{n\times s}, ..., x_{\ell+ n \times s}]$

as our input samples, where $x_t \in \mathbb{R}^p$ is the state of the system at time $t$. 

In [ ]:
def make_sequences(data, l, stride = 1):
    sequences = [data[i:i+l] for i in range(0, len(data) - l, stride)]
    return np.array(sequences)

#### TODO
 - Use the function above (`make_sequences`) to generate sequences. Use the sequence length of $\ell = 32$ and stride of 1 for healthy data (for training and validation) and stride of $\ell$ for faulty data (for testing).
 - Consider 80% of the healthy data for training and 20\% of it for validation.

**Note that we fit our model to the healthy data. After the training, we will employ the model to detect anomalies in the faulty time series.**

In [ ]:
l = 32
N_train = int(X_train[0].shape[0] * 0.8 / l)

X_val = []
y_val = []

####################
## YOUR CODE HERE ##
####################

In [ ]:
# print the shape of each split

####################
## YOUR CODE HERE ##
####################

#### TODO
Your data splits shapes should be something  like:
- Train 3 (125, 32, 7)
- Val 3 (31, 32, 7)
- Test 6 (155, 32, 7)

Write a function to have instead of 3 lists of 125 batches for train to (375, 32, 7) for train split.<br>
Same for other splits.

In [ ]:
def reshape_data(data):
    ...
    ####################
    ## YOUR CODE HERE ##
    ####################

X_train = reshape_data(X_train)
X_val = reshape_data(X_val)
X_test = reshape_data(X_test)

print("Train", X_train.shape)
print("Val", X_val.shape)
print("Test", X_test.shape)

In [ ]:
def plot_learning_curves(hist):
    epochs = np.arange(0, len(hist.history['loss'])) + 1
    sns.set(style='ticks')
    fig, ax = plt.subplots(1, 1, figsize = (5, 4), sharex=True)
    ax.plot(epochs, hist.history['loss'], label = 'Training loss', marker = 'o', ls = '--')
    ax.plot(epochs, hist.history['val_loss'], label = 'Validation loss', marker = 'o', ls = '--')

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Loss vs. Epoch')
    ax.legend()
    ax.set_yscale('log')
    sns.despine(trim=True, offset=5)

# Subtask 2.2

## MLP-based autoencoder

#### TODO
 - Consider the size of the bottleneck layer equal to 2.
 - Develop your model and print `model.summary()`.
 - Train the model.
 - Select the best model based on the validation loss.
 - Plot the learning curves.
 - Report the best validation loss you obtained.

In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
# plot_learning_curves(hist)

# Subtask 2.3

## CNN-based autoencoder

#### TODO
- Consider the size of the bottleneck layer equal to 2.

 - Develop your model and print `model.summary()`. Use 1D convolution, 1D pooling, and dense layers in the encoder and dense, 1D convolution transpose and 1D up-sampling layers in the decoder.
 - Train the model.
 - Select the best model based on the validation loss.
 - Plot the learning curves.
 - Report the best validation loss you obtained.

In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
# plot_learning_curves(hist)

# Subtask 2.4

## Visualize the data in the latent space

#### TODO
 - Take the encoder part of the model as a new model.
 - Use the encoder to map the test data to the latent space.
 - Plot the latent variables versus each other. 

 **Note:** Do not forget to add title, axis labels and a legend!
This applies in general, please keep in mind.

In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
####################
## YOUR CODE HERE ##
####################

# Subtask 2.5

## Size of the latent vector (bottleneck layer)

#### TODO
 - Use the CNN-based autoencoder.
 - Change the size of the bottleneck layer to 2, 4, 8, 16.
 - Train the model.
 - Select the best model based on the validation loss.
  - Report the best validation loss you obtained for each model.
 - Plot the best validation loss for each model versus the size of the bottleneck layer.

 **Note:** Do not forget to add title, axis labels and a legend!
This applies in general, please keep in mind.


In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
# plot_learning_curves(hist)

# Subtask 2.6

## Scheduled learning rate

When training a model, it is often useful to lower the learning rate as the training progresses. A learning rate schedule is a predefined mechanism that adjusts the learning rate between optimization steps during the training. [`tf.keras.optimizers.schedules`](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/schedules) provides several methods.


#### TODO
 - Consider the size of the bottleneck layer equal to 2.
 - Change the optimizer to use an exponential decay schedule: `tf.keras.optimizers.schedules.ExponentialDecay`.
 - Set the initial learning rate equal to 0.01.
 - Train the model and report the lowest validation loss.
 


In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
# plot_learning_curves(hist)

# Subtask 2.7

## Challenge

#### TODO
 - Use the CNN-based autoencoder.
 - Consider the size of the bottleneck layer equal to 8.
 - Modify the model architecture and the training process to obtain a better performance in reconstruction.
 - Select the best model based on the validation loss.
 - Plot the learning curves.
 - Report the best validation loss.

In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
####################
## YOUR CODE HERE ##
####################

# Subtask 2.8

## Reconstruction loss

#### TODO
 - Use the best model you have trained.
 - Compute the reconstruction loss for validation and test dataset.
    - You need to compute the loss for every time step.



In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
####################
## YOUR CODE HERE ##
####################

# Subtask 2.9

## Anomaly detection

#### TDOD
- Use the test dataset.
- Plot the ROC and precision-recall curves and report the area under the curve (AUC) for each of them.
- Select the threshold by allowing a false positive rate of 0.2.
- Detect anomalies using the threshold. 

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

####################
## YOUR CODE HERE ##
####################

In [ ]:
####################
## YOUR CODE HERE ##
####################

# Subtask 2.10

## Evaluate model performance

#### TODO
 - Compute accuracy, recall, precision f1-score.
 - Explain the effect of selecting different values for the threshold on the performance of the model.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
####################
## YOUR CODE HERE ##
####################

In [ ]:
####################
## YOUR CODE HERE ##
####################

<span style='color:red'>**Your answer:**</span>

...

## Visualize the detected anomalies

#### TDOD
 - Plot `Trq_MeanEff_Engine[Nm]` for the behavior `FP75`:
 - Visualize the healthy time steps by green.
 - Visualize the `GainRPMx2` true anomalous time steps by yellow, alpha: 0.3, and line width: 6.
 - Visualize the predicted anomalous time steps by red.

In [ ]:
####################
## YOUR CODE HERE ##
####################